# Parte B · Reproducción del artículo

**Artículo:** Moore, T. L. *et al.* *Unleashing the Power of Machine Learning in Nanomedicine Formulation Development.* **Adv. Funct. Mater.** 2026, 36, e14387.

## Qué se reproduce

| Resultado del paper | Qué contiene | Alcance |
|---|---|---|
| **Tabla 2** (resultado central) | Random Forest optimizado: RMSE y r² en entrenamiento y prueba | Lípidos y PLGA × dH, PdI, EE (6 modelos) |
| **Tablas S1 y S2** (SI) | 13 algoritmos con búsqueda aleatoria de 10 puntos y 10-fold CV: RMSE y r² medios | Lípidos (S1) y PLGA (S2); **11 de 13 algoritmos** |

## Cómo se reproduce

Es una implementación **propia en Python / scikit-learn** que sigue el código R del autor (`recursos/02_rf_lipid_model.R`, `03_rf_PLGA_model.R`, `04_AlgScreening.R`):

1. **Datos:** las tablas `dfML_lipid` / `dfML_PNP` emuladas en `00_preparacion_datos.ipynb`, sin limpieza adicional, igual que el autor.
2. **Validación, igual que el autor:**
   - partición aleatoria **por medición** 85/15, estratificada por cuartiles del objetivo, con semilla 1987;
   - 10-fold CV sobre el entrenamiento;
   - en los modelos de diámetro solo se usan las mediciones con PdI ≤ 0.25.
3. **Preprocesamiento, igual que el autor:** se eliminan las variables sin varianza (`step_zv`) y se estandarizan las X (`step_normalize`), ambas cosas estimadas dentro de cada fold.
4. **Métricas, igual que el autor:**
   - RMSE;
   - r² = correlación² entre observado y predicho (`yardstick::rsq` y `summary(lm(y ~ pred))$r.squared`);
   - además se reporta r² = 1 − SSE/SST, para comparar con la definición usual.

**Supuestos y desviaciones** (rejilla del RF reducida, algoritmos no implementados, equivalencias de hiperparámetros R ↔ Python): ver `docs/supuestos_y_validaciones.md`. Este notebook solo ejecuta y compara.

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy.stats import loguniform, randint, uniform, spearmanr
from sklearn.model_selection import KFold, ParameterSampler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Lasso, Ridge, LinearRegression, PoissonRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
import sklearn, xgboost

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATOS = ROOT / "datos_procesados"
SALIDA = ROOT / "resultados" / "parte_B"
SALIDA.mkdir(parents=True, exist_ok=True)
REF_R = ROOT / "resultados" / "referencia_R"

# Constantes del autor (recursos/01_mfNP.R)
SEED = 1987          # set.seed(seed)
PDI_LIM = 0.250      # pdi_lim
TRAIN_PERC = 0.85    # trainPerc
VFOLDS = 10          # vfolds
N_GRID = 10          # nGrid (puntos de la búsqueda aleatoria en 04_AlgScreening.R)
N_JOBS = 6           # procesos en paralelo; subir (p. ej. -1 = todos los núcleos) solo si hay memoria libre

# Estilo de figuras: paleta categórica validada (slots 1-3), texto en tinta neutra
C_TRAIN, C_TEST, C_EXTRA = "#2a78d6", "#eb6834", "#1baf7a"
TINTA, TINTA2, REJILLA = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 10,
    "axes.edgecolor": TINTA2, "axes.labelcolor": TINTA, "xtick.color": TINTA2, "ytick.color": TINTA2,
    "axes.grid": True, "grid.color": REJILLA, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})
print(f"scikit-learn {sklearn.__version__} | xgboost {xgboost.__version__} | pandas {pd.__version__} | numpy {np.__version__}")

## 1. Datos y conjuntos de modelado

Para cada familia y objetivo, el autor arma un conjunto distinto:
- los predictores son todas las columnas excepto `payload`, los demás objetivos y `zetaPot_mV`;
- en los modelos de diámetro (dH) se filtra antes PdI ≤ 0.25.

Las columnas `meta_*` (identificador de formulación, tipo de partícula, lote) **no son predictores**.

In [ ]:
ML = {
    "lipidos": pd.read_csv(DATOS / "ml_lipidos.csv"),
    "plga": pd.read_csv(DATOS / "ml_plga.csv"),
}
FAMILIAS = {"lipidos": "Lípidos", "plga": "PLGA"}
OBJETIVOS = {"hdiam_nm": "dH (nm)", "pdi": "PdI", "percEE": "EE (%)"}
NO_PREDICTORES = ["payload", "hdiam_nm", "pdi", "zetaPot_mV", "percEE"]


def predictores(familia):
    d = ML[familia]
    return [c for c in d.columns if c not in NO_PREDICTORES and not c.startswith("meta_")]


def conjunto(familia, objetivo):
    # Datos que recibe cada modelo (con el índice original de la tabla procesada)
    d = ML[familia]
    if objetivo == "hdiam_nm":
        d = d[d["pdi"] <= PDI_LIM]          # 02/03: dfML[which(dfML$pdi <= pdi_lim), ]
    return d


for fam in FAMILIAS:
    print(f"{FAMILIAS[fam]}: {len(predictores(fam))} predictores → {predictores(fam)}")

## 2. Partición train/test y folds de validación cruzada

**Emulación de `rsample::initial_split(data, prop = 0.85, strata = <objetivo>)`:**
- el objetivo se divide en **cuartiles** (`breaks = 4`);
- dentro de cada estrato se toma al azar el `floor(n · 0.85)` para entrenamiento;
- los estratos con menos del 10 % de los datos (`pool = 0.1`) se fusionan con su vecino.

**Emulación de `vfold_cv(train, v = 10)`:** 10 folds aleatorios sobre las filas de entrenamiento, sin estratificar.

Como R y Python usan generadores aleatorios distintos, la semilla 1987 **no produce las mismas filas** que en el paper. Ese efecto se aísla en la sección 4 con la partición exportada desde R.

> Importante para la Parte C: igual que en el paper, la partición es **por medición**. Las réplicas de una misma formulación pueden quedar repartidas entre train y test.

In [ ]:
def estratos_rsample(y, cortes=4, pool=0.10):
    # Cuartiles del objetivo; los estratos pequeños (< pool) se fusionan con el vecino
    e = np.asarray(pd.qcut(y, q=cortes, labels=False, duplicates="drop"))
    while True:
        etiquetas, n = np.unique(e, return_counts=True)
        if len(etiquetas) <= 1 or n.min() >= pool * len(y):
            return e
        i = int(np.argmin(n))
        vecino = etiquetas[i - 1] if i > 0 else etiquetas[i + 1]
        e[e == etiquetas[i]] = vecino


def particion_python(familia, objetivo):
    d = conjunto(familia, objetivo)
    y = d[objetivo].to_numpy()
    rng = np.random.default_rng(SEED)          # set.seed(seed) antes de cada initial_split
    estratos = estratos_rsample(y)
    en_train = np.zeros(len(y), dtype=bool)
    for e in np.unique(estratos):
        idx = np.flatnonzero(estratos == e)
        en_train[rng.choice(idx, int(np.floor(len(idx) * TRAIN_PERC)), replace=False)] = True
    part = pd.DataFrame({"conjunto": np.where(en_train, "train", "test"), "fold": -1}, index=d.index)
    kf = KFold(n_splits=VFOLDS, shuffle=True, random_state=SEED)
    idx_train = part.index[en_train]
    for k, (_, va) in enumerate(kf.split(idx_train)):
        part.loc[idx_train[va], "fold"] = k
    return part


PARTICIONES = {(f, o): particion_python(f, o) for f in FAMILIAS for o in OBJETIVOS}

filas = []
for (f, o), p in PARTICIONES.items():
    filas.append({"familia": FAMILIAS[f], "objetivo": OBJETIVOS[o], "n total": len(p),
                  "n train": int((p.conjunto == "train").sum()), "n test": int((p.conjunto == "test").sum())})
tabla_n = pd.DataFrame(filas)
tabla_n

In [ ]:
# Se guardan las particiones (las usa la Parte C para medir la fuga por réplicas)
registros = []
for (f, o), p in PARTICIONES.items():
    r = p.copy()
    r["indice"] = r.index
    r["familia"], r["objetivo"] = f, o
    r["meta_formulation_id"] = ML[f].loc[r.index, "meta_formulation_id"].to_numpy()
    registros.append(r)
pd.concat(registros)[["familia", "objetivo", "indice", "conjunto", "fold", "meta_formulation_id"]] \
  .to_csv(SALIDA / "particiones_python.csv", index=False)
print("Guardado:", SALIDA / "particiones_python.csv")

## 3. Herramientas comunes: tubería, métricas y búsqueda de hiperparámetros

- **Tubería (equivale a la *recipe* del autor):** `VarianceThreshold(0)` (`step_zv`) → `StandardScaler` (`step_normalize`) → modelo. Al ajustarse dentro de cada fold, el escalado no filtra información.
- **Métricas por fold:** RMSE y r² (correlación²), como `yardstick`. Se resumen con la media y el error estándar (`std_err` de `tune::show_best`).
- **Selección:** la configuración con **menor RMSE medio** (`select_best(metric = "rmse")`).

In [ ]:
def tuberia(modelo, escalar_y=False):
    pipe = Pipeline([("zv", VarianceThreshold(0.0)), ("esc", StandardScaler()), ("modelo", modelo)])
    # kernlab::ksvm (SVM del autor) escala también la variable respuesta (scaled = TRUE)
    return TransformedTargetRegressor(regressor=pipe, transformer=StandardScaler()) if escalar_y else pipe


def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))


def rsq_cor(y, p):
    # r² como lo calcula el autor: correlación de Pearson al cuadrado
    # (yardstick::rsq y summary(lm(y ~ pred))$r.squared). NaN si la predicción es constante.
    y, p = np.asarray(y, float), np.asarray(p, float)
    if np.std(p) < 1e-12 or np.std(y) < 1e-12:
        return np.nan
    return float(np.corrcoef(y, p)[0, 1] ** 2)


def evaluar_configuraciones(fabrica, configuraciones, X, y, folds):
    # 10-fold CV de todas las configuraciones, en paralelo sobre (configuración × fold)
    def una(i, k):
        tr, va = folds != k, folds == k
        m = fabrica(configuraciones[i], X[tr], y[tr])
        m.fit(X[tr], y[tr])
        p = m.predict(X[va])
        return i, k, rmse(y[va], p), rsq_cor(y[va], p)

    res = Parallel(n_jobs=N_JOBS)(delayed(una)(i, k) for i in range(len(configuraciones)) for k in range(VFOLDS))
    res = pd.DataFrame(res, columns=["config", "fold", "rmse", "rsq"])
    resumen = res.groupby("config").agg(
        rmse_media=("rmse", "mean"), rmse_se=("rmse", lambda s: s.std(ddof=1) / np.sqrt(s.count())),
        rsq_media=("rsq", "mean"), rsq_se=("rsq", lambda s: s.std(ddof=1) / np.sqrt(s.count())))
    resumen["params"] = [configuraciones[i] for i in resumen.index]
    return resumen


def matrices(familia, objetivo, part):
    d = conjunto(familia, objetivo)
    X = d[predictores(familia)].to_numpy(float)
    y = d[objetivo].to_numpy(float)
    tr = (part.loc[d.index, "conjunto"] == "train").to_numpy()
    folds = part.loc[d.index, "fold"].to_numpy()[tr]
    return X[tr], y[tr], X[~tr], y[~tr], folds, d.index[tr], d.index[~tr]

## 4. Tabla 2: Random Forest optimizado

**Lo que hace el autor** (`02_rf_lipid_model.R` / `03_rf_PLGA_model.R`):
- `rand_forest(mtry, trees, min_n)` con el motor `randomForest`;
- rejilla regular de **10 × 10 × 10 = 1000 combinaciones** (`grid_regular(levels = 10)`) con 10-fold CV;
- se elige el mejor RMSE, se reentrena con todo el entrenamiento y se evalúa en train y test.

**Lo que se hace aquí (rejilla reducida, ver `docs/supuestos_y_validaciones.md`):**
- `trees` = **500 fijo**. El paper reporta que el error no mejora con más de ~250 árboles (Fig. S3).
- `mtry` toma **todos** los valores de 1 a p (incluye los 10 niveles del autor).
- `min_n` toma los **mismos 10 niveles** del autor (2 a 40).
- `min_n` (`nodesize` de randomForest: un nodo se divide solo si tiene más de `nodesize` observaciones) se traduce a `min_samples_split = min_n + 1`.

In [ ]:
MIN_N_NIVELES = np.round(np.linspace(2, 40, 10)).astype(int)     # niveles de grid_regular para min_n
N_ARBOLES = 500


def fabrica_rf(params, X_tr, y_tr):
    p = X_tr.shape[1]
    return tuberia(RandomForestRegressor(
        n_estimators=params.get("trees", N_ARBOLES), max_features=min(params["mtry"], p),
        min_samples_split=params["min_n"] + 1, random_state=SEED, n_jobs=1))


def ajustar_rf(familia, objetivo, part):
    X_tr, y_tr, X_te, y_te, folds, i_tr, i_te = matrices(familia, objetivo, part)
    p = X_tr.shape[1]
    rejilla = [{"mtry": m, "min_n": int(n)} for m in range(1, p + 1) for n in MIN_N_NIVELES]
    cv = evaluar_configuraciones(fabrica_rf, rejilla, X_tr, y_tr, folds)
    mejor = cv.sort_values("rmse_media").iloc[0]
    modelo = fabrica_rf(mejor["params"], X_tr, y_tr).fit(X_tr, y_tr)
    p_tr, p_te = modelo.predict(X_tr), modelo.predict(X_te)
    fila = {
        "familia": familia, "objetivo": objetivo, "n_train": len(y_tr), "n_test": len(y_te),
        "mtry": mejor["params"]["mtry"], "min_n": mejor["params"]["min_n"],
        "cv_rmse": mejor["rmse_media"], "cv_rmse_se": mejor["rmse_se"], "cv_rsq": mejor["rsq_media"],
        "rmse_train": rmse(y_tr, p_tr), "rmse_test": rmse(y_te, p_te),
        "rsq_train": rsq_cor(y_tr, p_tr), "rsq_test": rsq_cor(y_te, p_te),
        "r2sse_train": r2_score(y_tr, p_tr), "r2sse_test": r2_score(y_te, p_te),
    }
    pred = pd.DataFrame({"indice": np.r_[i_tr, i_te], "y": np.r_[y_tr, y_te], "pred": np.r_[p_tr, p_te],
                         "conjunto": ["train"] * len(y_tr) + ["test"] * len(y_te)})
    return fila, pred, cv


t0 = time.time()
RF_FILAS, RF_PRED, RF_CV = [], {}, {}
for (f, o), part in PARTICIONES.items():
    fila, pred, cv = ajustar_rf(f, o, part)
    RF_FILAS.append(fila); RF_PRED[(f, o)] = pred; RF_CV[(f, o)] = cv
    print(f"{FAMILIAS[f]:8s} {OBJETIVOS[o]:8s} mejor mtry={fila['mtry']:2d}, min_n={fila['min_n']:2d} "
          f"| RMSE test={fila['rmse_test']:.3f}, r² test={fila['rsq_test']:.3f}")
print(f"Tiempo: {time.time() - t0:.0f} s")
rf_python = pd.DataFrame(RF_FILAS)

# Predicciones fila a fila (las usa la Parte C para comparar con el "memorizador")
pd.concat([p.assign(familia=f, objetivo=o) for (f, o), p in RF_PRED.items()])   .to_csv(SALIDA / "predicciones_rf_python.csv", index=False)

### Comparación con la Tabla 2 del paper

Valores del paper transcritos de la Tabla 2 (*Adv. Funct. Mater.* 2026, e14387). La columna **Δ** es Python − paper.

In [ ]:
TABLA2_PAPER = pd.DataFrame([
    # familia, objetivo, RMSE train, RMSE test, r² train, r² test
    ("lipidos", "hdiam_nm", 9.8, 9.5, 0.950, 0.922),
    ("lipidos", "pdi", 0.046, 0.053, 0.903, 0.787),
    ("lipidos", "percEE", 12.9, 10.6, 0.668, 0.472),
    ("plga", "hdiam_nm", 5.4, 7.5, 0.985, 0.964),
    ("plga", "pdi", 0.033, 0.047, 0.849, 0.560),
    ("plga", "percEE", 3.1, 2.2, 0.970, 0.984),
], columns=["familia", "objetivo", "paper_rmse_train", "paper_rmse_test", "paper_rsq_train", "paper_rsq_test"])


def tabla_comparativa(rf):
    t = rf.merge(TABLA2_PAPER, on=["familia", "objetivo"])
    t["Δ rmse_test"] = t["rmse_test"] - t["paper_rmse_test"]
    t["Δ rsq_test"] = t["rsq_test"] - t["paper_rsq_test"]
    t.insert(0, "Familia", t.pop("familia").map(FAMILIAS))
    t.insert(1, "Objetivo", t.pop("objetivo").map(OBJETIVOS))
    return t


comp_python = tabla_comparativa(rf_python)
comp_python.to_csv(SALIDA / "tabla2_rf_python.csv", index=False)
vista = comp_python[["Familia", "Objetivo", "rmse_train", "paper_rmse_train", "rmse_test", "paper_rmse_test",
                     "Δ rmse_test", "rsq_train", "paper_rsq_train", "rsq_test", "paper_rsq_test", "Δ rsq_test",
                     "r2sse_test"]]
vista.round(3)

In [ ]:
fig, ejes = plt.subplots(2, 3, figsize=(12, 7.5))
for i, f in enumerate(FAMILIAS):
    for j, o in enumerate(OBJETIVOS):
        ax, pr = ejes[i, j], RF_PRED[(f, o)]
        fila = rf_python[(rf_python.familia == f) & (rf_python.objetivo == o)].iloc[0]
        lim = [min(pr.y.min(), pr.pred.min()), max(pr.y.max(), pr.pred.max())]
        ax.plot(lim, lim, color=TINTA2, lw=1, ls="--", zorder=1)
        for conj, color, nombre in [("train", C_TRAIN, "entrenamiento"), ("test", C_TEST, "prueba")]:
            s = pr[pr.conjunto == conj]
            ax.scatter(s.y, s.pred, s=22 if conj == "train" else 34, color=color, alpha=0.75,
                       edgecolor="white", linewidth=0.6, label=nombre, zorder=2 if conj == "train" else 3)
        ax.set_title(f"{FAMILIAS[f]} · {OBJETIVOS[o]}", color=TINTA, fontsize=11)
        ax.text(0.03, 0.97, f"test: r² = {fila.rsq_test:.3f}\nRMSE = {fila.rmse_test:.3g}",
                transform=ax.transAxes, va="top", color=TINTA, fontsize=9)
        ax.set_xlabel("Observado"); ax.set_ylabel("Predicho")
ejes[0, 0].legend(loc="lower right")
fig.suptitle("Random Forest (reproducción en Python): observado vs predicho", color=TINTA, fontsize=13)
fig.tight_layout()
fig.savefig(SALIDA / "fig_tabla2_observado_vs_predicho.png", bbox_inches="tight")
plt.show()

### Variante: misma partición que el código R del autor

Si el notebook `00R_referencia_autor.ipynb` ya se ejecutó, en `resultados/referencia_R/` hay dos archivos:
- **las filas exactas** de train/test y de cada fold que generó R con `set.seed(1987)`;
- **las métricas** que obtuvo el código del autor.

Al repetir el RF con esa misma partición se separa el efecto del **generador aleatorio** (qué filas caen en el test) del efecto de **la implementación** (R vs Python, rejilla reducida). Si los archivos no existen, esta sección se omite.

In [ ]:
RUTA_PART_R = REF_R / "particiones_R.csv"
RUTA_TABLA2_R = REF_R / "tabla2_rf_R.csv"

if RUTA_PART_R.exists():
    pr_r = pd.read_csv(RUTA_PART_R)
    filas_r = []
    for (f, o) in PARTICIONES:
        s = pr_r[(pr_r.familia == f) & (pr_r.objetivo == o)].set_index("indice")
        part_r = s[["conjunto", "fold"]].copy()
        fila, _, _ = ajustar_rf(f, o, part_r)
        filas_r.append(fila)
    comp_misma = tabla_comparativa(pd.DataFrame(filas_r))
    if RUTA_TABLA2_R.exists():
        t_r = pd.read_csv(RUTA_TABLA2_R)
        t_r["Familia"] = t_r.familia.map(FAMILIAS); t_r["Objetivo"] = t_r.objetivo.map(OBJETIVOS)
        comp_misma = comp_misma.merge(t_r.drop(columns=["familia", "objetivo"]).add_prefix("R_")
                                      .rename(columns={"R_Familia": "Familia", "R_Objetivo": "Objetivo"}),
                                      on=["Familia", "Objetivo"], how="left")
    comp_misma.to_csv(SALIDA / "tabla2_rf_python_particionR.csv", index=False)
    display(comp_misma.round(3))
else:
    print("No existe resultados/referencia_R/particiones_R.csv → se omite esta variante.\n"
          "Ejecutar notebooks/00R_referencia_autor.ipynb (requiere R) y volver a correr esta celda.")

## 5. Tablas S1 y S2: comparación de algoritmos

**Lo que hace el autor** (`04_AlgScreening.R`):
- para cada algoritmo, una **búsqueda aleatoria de 10 configuraciones** (`grid_random(size = 10)`, `set.seed(1987)`);
- sobre **los mismos folds** del entrenamiento del RF;
- reporta el RMSE medio (± `std_err`) del mejor RMSE y el r² medio del mejor r² (`show_best(metric = "rsq")[1, ]`, que puede ser **otra** configuración).

**Equivalencias usadas** (rangos por defecto de `dials`; detalle en `docs/supuestos_y_validaciones.md`):

| Algoritmo (paper) | R (motor) | Python | Hiperparámetros muestreados |
|---|---|---|---|
| bagged trees | `bag_tree` (`rpart`, 11 bolsas) | `BaggingRegressor(DecisionTree, 11)` | tree_depth 1–15, min_n 2–40, cost_complexity 10⁻¹⁰–10⁻¹ (× Var(y)) |
| xgboost | `boost_tree` (`xgboost`) | `XGBRegressor` | mtry, trees 1–2000, min_n 2–40, tree_depth 1–15, learn_rate 10⁻¹⁰–10⁻¹, loss_reduction 10⁻¹⁰–10^1.5, sample_size 0.1–1 |
| k-nearest neighbors | `nearest_neighbor` (`kknn`) | `KNeighborsRegressor` | neighbors 1–10, pesos uniforme/inverso de la distancia, dist_power 1–2 |
| LASSO | `glmnet` (mixture = 1) | `Lasso` | penalty 10⁻¹⁰–1 |
| linear regression | `lm` | `LinearRegression` | (sin hiperparámetros) |
| Poisson regression | `glm` (Poisson) | `PoissonRegressor(alpha = 0)` | (sin hiperparámetros) |
| random forest | `randomForest` | `RandomForestRegressor` | mtry, trees 1–2000, min_n 2–40 |
| RIDGE | `glmnet` (mixture = 0) | `Ridge(alpha = n·λ)` | penalty 10⁻¹⁰–1 |
| SVM linear / polynomial / radial | `kernlab` (y escalada) | `SVR` + y escalada | cost 2⁻¹⁰–2⁵, margin 0–0.2, degree 1–3, scale_factor 10⁻¹⁰–10⁻¹, rbf_sigma 10⁻¹⁰–1 |
| **bagged MARS** | `earth` | **no implementado** | — |
| **bagged MLP** | `nnet` | **no implementado** | — |

In [ ]:
def fab_bagtree(c, X, y):
    return tuberia(BaggingRegressor(
        DecisionTreeRegressor(max_depth=c["tree_depth"], min_samples_split=c["min_n"],
                              ccp_alpha=c["cost_complexity"] * np.var(y)),
        n_estimators=11, random_state=SEED, n_jobs=1))


def fab_xgb(c, X, y):
    p = X.shape[1]
    return tuberia(XGBRegressor(
        n_estimators=c["trees"], max_depth=c["tree_depth"], min_child_weight=c["min_n"],
        learning_rate=c["learn_rate"], gamma=c["loss_reduction"], subsample=c["sample_size"],
        colsample_bynode=min(c["mtry"], p) / p, random_state=SEED, n_jobs=1, verbosity=0))


def fab_knn(c, X, y):
    return tuberia(KNeighborsRegressor(n_neighbors=min(c["neighbors"], len(y)), weights=c["weight_func"],
                                       p=c["dist_power"]))


def fab_lasso(c, X, y):
    return tuberia(Lasso(alpha=c["penalty"], max_iter=100000))


def fab_ridge(c, X, y):
    return tuberia(Ridge(alpha=len(y) * c["penalty"]))


def fab_lm(c, X, y):
    return tuberia(LinearRegression())


def fab_poisson(c, X, y):
    return tuberia(PoissonRegressor(alpha=0.0, max_iter=10000))


def fab_svm_lin(c, X, y):
    return tuberia(SVR(kernel="linear", C=c["cost"], epsilon=c["margin"], max_iter=1_000_000), escalar_y=True)


def fab_svm_poly(c, X, y):
    return tuberia(SVR(kernel="poly", C=c["cost"], epsilon=c["margin"], degree=c["degree"],
                       gamma=c["scale_factor"], coef0=1.0, max_iter=1_000_000), escalar_y=True)


def fab_svm_rbf(c, X, y):
    return tuberia(SVR(kernel="rbf", C=c["cost"], epsilon=c["margin"], gamma=c["rbf_sigma"],
                       max_iter=1_000_000), escalar_y=True)


def fab_rf_rand(c, X, y):
    return fabrica_rf(c, X, y)


SVM_COMUN = {"cost": loguniform(2.0 ** -10, 2.0 ** 5), "margin": uniform(0, 0.2)}
ALGORITMOS = {
    # nombre del paper: (fábrica, distribuciones de la búsqueda aleatoria; None = sin hiperparámetros)
    "bagged trees": (fab_bagtree, {"tree_depth": randint(1, 16), "min_n": randint(2, 41),
                                   "cost_complexity": loguniform(1e-10, 1e-1)}),
    "k-nearest neighbors": (fab_knn, {"neighbors": randint(1, 11), "weight_func": ["uniform", "distance"],
                                      "dist_power": uniform(1, 1)}),
    "LASSO": (fab_lasso, {"penalty": loguniform(1e-10, 1)}),
    "linear regression": (fab_lm, None),
    "Poisson regression": (fab_poisson, None),
    "random forest": (fab_rf_rand, {"mtry": "p", "trees": randint(1, 2001), "min_n": randint(2, 41)}),
    "RIDGE": (fab_ridge, {"penalty": loguniform(1e-10, 1)}),
    "SVM, linear": (fab_svm_lin, dict(SVM_COMUN)),
    "SVM, polynomial": (fab_svm_poly, {**SVM_COMUN, "degree": randint(1, 4),
                                       "scale_factor": loguniform(1e-10, 1e-1)}),
    "SVM, radial": (fab_svm_rbf, {**SVM_COMUN, "rbf_sigma": loguniform(1e-10, 1)}),
    "xgboost": (fab_xgb, {"mtry": "p", "trees": randint(1, 2001), "min_n": randint(2, 41),
                          "tree_depth": randint(1, 16), "learn_rate": loguniform(1e-10, 1e-1),
                          "loss_reduction": loguniform(1e-10, 10 ** 1.5), "sample_size": uniform(0.1, 0.9)}),
}
NO_IMPLEMENTADOS = ["bagged MARS", "bagged MLP"]


def configuraciones(dist, p):
    if dist is None:
        return [{}]
    dist = {k: (randint(1, p + 1) if v == "p" else v) for k, v in dist.items()}
    return list(ParameterSampler(dist, n_iter=N_GRID, random_state=SEED))

In [ ]:
t0 = time.time()
filas_cv = []
for (f, o), part in PARTICIONES.items():
    X_tr, y_tr, _, _, folds, _, _ = matrices(f, o, part)
    for nombre, (fab, dist) in ALGORITMOS.items():
        cv = evaluar_configuraciones(fab, configuraciones(dist, X_tr.shape[1]), X_tr, y_tr, folds)
        mejor_rmse = cv.sort_values("rmse_media").iloc[0]
        mejor_rsq = cv.sort_values("rsq_media", ascending=False).iloc[0]
        filas_cv.append({"familia": f, "objetivo": o, "algoritmo": nombre,
                         "rmse": mejor_rmse.rmse_media, "rmse_se": mejor_rmse.rmse_se,
                         "rsq": mejor_rsq.rsq_media, "rsq_se": mejor_rsq.rsq_se,
                         "rsq_de_config_rmse": mejor_rmse.rsq_media})
    print(f"{FAMILIAS[f]:8s} {OBJETIVOS[o]:8s} listo ({time.time() - t0:.0f} s)")
cv_python = pd.DataFrame(filas_cv)

### Comparación con las Tablas S1 (lípidos) y S2 (PLGA)

Valores transcritos del SI final (`contexto/adfm71140-sup-0001-suppmat.pdf`, págs. 4–5). El SI dice "media ± 1 SD", pero el código del autor guarda el `std_err` de `show_best()`. Aquí se compara solo la **media**.

In [ ]:
# (rmse, r²) por algoritmo: [dH, PdI, EE]
S1 = {
    "bagged MARS": [(24, .678), (.082, .598), (11.1, .908)],
    "bagged MLP": [(20, .756), (.070, .734), (10.3, .921)],
    "bagged trees": [(16, .858), (.069, .758), (10.4, .918)],
    "k-nearest neighbors": [(17, .844), (.070, .755), (9.8, .928)],
    "LASSO": [(36, .325), (.084, .573), (11.5, .900)],
    "linear regression": [(36, .325), (.084, .572), (11.5, .900)],
    "Poisson regression": [(37, .305), (.079, .619), (11.3, .903)],
    "random forest": [(16, .860), (.067, .749), (9.8, .929)],
    "RIDGE": [(36, .327), (.083, .574), (11.6, .900)],
    "SVM, linear": [(38, .277), (.084, .562), (11.6, .899)],
    "SVM, polynomial": [(39, .349), (.097, .554), (13, .902)],
    "SVM, radial": [(31, .483), (.070, .708), (11.3, .897)],
    "xgboost": [(17, .832), (.070, .725), (10.8, .914)],
}
S2 = {
    "bagged MARS": [(15, .884), (.066, .379), (10.7, .911)],
    "bagged MLP": [(13, .913), (.052, .613), (4.6, .983)],
    "bagged trees": [(11, .936), (.053, .622), (3.9, .987)],
    "k-nearest neighbors": [(11, .932), (.051, .661), (3.8, .986)],
    "LASSO": [(15, .882), (.076, .182), (12.4, .884)],
    "linear regression": [(15, .882), (.076, .182), (12.4, .884)],
    "random forest": [(11, .935), (.053, .629), (3.9, .987)],
    "Poisson regression": [(16, .870), (.075, .199), (11.0, .905)],
    "RIDGE": [(16, .882), (.076, .182), (12.4, .883)],
    "SVM, linear": [(15, .883), (.076, .175), (12.6, .884)],
    "SVM, polynomial": [(23, .855), (.078, .165), (12.6, .881)],
    "SVM, radial": [(12, .912), (.065, .395), (8.9, .939)],
    "xgboost": [(11, .934), (.061, .503), (4.0, .986)],
}
filas = []
for fam, tabla in [("lipidos", S1), ("plga", S2)]:
    for alg, vals in tabla.items():
        for o, (r, q) in zip(OBJETIVOS, vals):
            filas.append({"familia": fam, "objetivo": o, "algoritmo": alg, "paper_rmse": r, "paper_rsq": q})
SI_PAPER = pd.DataFrame(filas)

comp_cv = SI_PAPER.merge(cv_python, on=["familia", "objetivo", "algoritmo"], how="left")
comp_cv["estado"] = np.where(comp_cv.algoritmo.isin(NO_IMPLEMENTADOS), "no implementado", "reproducido")
comp_cv["Δ rmse"] = comp_cv["rmse"] - comp_cv["paper_rmse"]
comp_cv["Δ rsq"] = comp_cv["rsq"] - comp_cv["paper_rsq"]
comp_cv.to_csv(SALIDA / "tablaS1_S2_algoritmos_python.csv", index=False)

for fam, titulo in [("lipidos", "Tabla S1 · Lípidos"), ("plga", "Tabla S2 · PLGA")]:
    t = comp_cv[comp_cv.familia == fam].copy()
    t["obj"] = t.objetivo.map(OBJETIVOS)
    ancha = t.pivot_table(index="algoritmo", columns="obj",
                          values=["paper_rmse", "rmse", "paper_rsq", "rsq"], aggfunc="first")
    ancha = ancha.reindex(columns=[(m, ob) for ob in OBJETIVOS.values() for m in ["paper_rmse", "rmse", "paper_rsq", "rsq"]])
    print(titulo)
    display(ancha.round(3))

In [ ]:
# ¿Se conserva el ranking de algoritmos? Correlación de Spearman entre el RMSE de Python y el del paper
filas = []
for (f, o), g in comp_cv[comp_cv.estado == "reproducido"].groupby(["familia", "objetivo"]):
    rho = spearmanr(g["rmse"], g["paper_rmse"]).statistic
    mejor_py = g.sort_values("rmse").algoritmo.iloc[0]
    mejor_paper = g.sort_values(["paper_rmse", "paper_rsq"], ascending=[True, False]).algoritmo.iloc[0]
    rf_pos = int(g.sort_values("rmse").algoritmo.tolist().index("random forest")) + 1
    filas.append({"Familia": FAMILIAS[f], "Objetivo": OBJETIVOS[o], "ρ Spearman (RMSE)": rho,
                  "Mejor en Python": mejor_py, "Mejor en el paper": mejor_paper,
                  "Puesto del RF en Python (de 11)": rf_pos})
ranking = pd.DataFrame(filas)
ranking.to_csv(SALIDA / "ranking_algoritmos.csv", index=False)
ranking.round(3)

In [ ]:
fig, ejes = plt.subplots(2, 3, figsize=(12, 7.5))
rep = comp_cv[comp_cv.estado == "reproducido"]
for i, f in enumerate(FAMILIAS):
    for j, o in enumerate(OBJETIVOS):
        ax = ejes[i, j]
        g = rep[(rep.familia == f) & (rep.objetivo == o)]
        ax.plot([0, 1], [0, 1], color=TINTA2, lw=1, ls="--", zorder=1)
        es_rf = g.algoritmo == "random forest"
        ax.scatter(g.paper_rsq[~es_rf], g.rsq[~es_rf], s=40, color=C_TRAIN, edgecolor="white",
                   linewidth=0.6, label="otros algoritmos", zorder=2)
        ax.scatter(g.paper_rsq[es_rf], g.rsq[es_rf], s=60, color=C_TEST, edgecolor="white",
                   linewidth=0.8, label="random forest", zorder=3)
        lo = float(np.nanmin([g.paper_rsq.min(), g.rsq.min()])) - 0.05
        ax.set_xlim(max(lo, -0.05), 1.01); ax.set_ylim(max(lo, -0.05), 1.01)
        ax.set_title(f"{FAMILIAS[f]} · {OBJETIVOS[o]}", color=TINTA, fontsize=11)
        ax.set_xlabel("r² CV del paper (Tablas S1/S2)"); ax.set_ylabel("r² CV en Python")
ejes[0, 0].legend(loc="lower right")
fig.suptitle("Comparación de algoritmos: r² de validación cruzada, Python vs paper", color=TINTA, fontsize=13)
fig.tight_layout()
fig.savefig(SALIDA / "fig_tablaS1_S2_r2_python_vs_paper.png", bbox_inches="tight")
plt.show()

In [ ]:
# Si existe la corrida R de referencia, se añaden sus métricas de CV (incluye bagged MARS y bagged MLP)
RUTA_CV_R = REF_R / "cv_algoritmos_R.csv"
if RUTA_CV_R.exists():
    cv_r = pd.read_csv(RUTA_CV_R).rename(columns={"rmse": "R_rmse", "rsq": "R_rsq"})
    comp_cv_r = comp_cv.merge(cv_r[["familia", "objetivo", "algoritmo", "R_rmse", "R_rsq"]],
                              on=["familia", "objetivo", "algoritmo"], how="left")
    comp_cv_r.to_csv(SALIDA / "tablaS1_S2_algoritmos_python_vs_R.csv", index=False)
    display(comp_cv_r[["familia", "objetivo", "algoritmo", "paper_rmse", "R_rmse", "rmse",
                       "paper_rsq", "R_rsq", "rsq"]].round(3))
else:
    print("No existe resultados/referencia_R/cv_algoritmos_R.csv → la comparación con R queda pendiente "
          "(ver docs/supuestos_y_validaciones.md).")

## 6. Resumen numérico de discrepancias

In [ ]:
resumen = comp_python[["Familia", "Objetivo", "rmse_test", "paper_rmse_test", "rsq_test", "paper_rsq_test", "r2sse_test"]].copy()
resumen["error relativo RMSE test (%)"] = 100 * (resumen.rmse_test - resumen.paper_rmse_test) / resumen.paper_rmse_test
resumen["|Δ r² test|"] = (resumen.rsq_test - resumen.paper_rsq_test).abs()
print("Tabla 2 · Random Forest (prueba)")
display(resumen.round(3))

rep = comp_cv[comp_cv.estado == "reproducido"]
print("Tablas S1/S2 · diferencia media absoluta de r² de CV por objetivo (11 algoritmos)")
display(rep.assign(abs_drsq=rep["Δ rsq"].abs()).groupby(["familia", "objetivo"])["abs_drsq"]
        .agg(["mean", "max"]).rename(index=FAMILIAS, level=0).rename(index=OBJETIVOS, level=1).round(3))

## 7. Análisis de discrepancias

> Corresponde a la ejecución con semilla 1987, 6 procesos y las versiones de `requirements.txt`, comparada con la **corrida del código R original del autor** (`00R_referencia_autor.ipynb`, 17-nov-2026, `tune` 1.3.0). Los supuestos están en `docs/supuestos_y_validaciones.md` y la explicación completa en `docs/reproduccion_paper.md`.

### 7.1 Tabla 2: tres comparaciones para separar las causas

| Modelo | r² test **paper** | r² test **código R del autor** | r² test **Python, misma partición que R** | r² test **Python, partición propia** |
|---|---|---|---|---|
| Lípidos · dH | 0.922 | 0.834 | 0.839 | 0.817 |
| Lípidos · PdI | 0.787 | 0.869 | 0.868 | 0.856 |
| Lípidos · EE | 0.472 | 0.951 | 0.957 | 0.827 |
| PLGA · dH | 0.964 | 0.945 | 0.944 | 0.957 |
| PLGA · PdI | 0.560 | 0.607 | 0.590 | 0.703 |
| PLGA · EE | 0.984 | 0.976 | 0.977 | 0.992 |

**Lo que muestra cada comparación:**

1. **Implementación (Python vs R con las mismas filas): prácticamente idénticas.**
   - Con la partición exacta de R, la reimplementación en Python da el mismo resultado que el código del autor: r² de prueba con diferencias de 0.001 a 0.017 y r² de entrenamiento con diferencias ≤ 0.003.
   - Esto vale **aunque Python use la rejilla reducida** (500 árboles fijos) y el autor la completa: la simplificación no cambia los resultados. Validación §1.1 de `docs/`.
2. **Azar (partición propia vs partición de R): diferencias moderadas.**
   - R y Python, con la misma semilla, eligen filas de prueba distintas: solo comparten entre 1 y 13 filas por modelo.
   - Cambiar de partición mueve el r² de prueba hasta ~0.13 (lípidos · EE: 0.827 vs 0.957; PLGA · PdI: 0.703 vs 0.590).
   - Con conjuntos de prueba de 24–52 filas, **una sola partición no basta para comparar con precisión de centésimas**.
3. **El paper frente al código actual del autor: hay diferencias que no se explican.**
   - El propio código del autor, ejecutado hoy, **no reproduce la Tabla 2 publicada** en lípidos · dH (0.834 vs 0.922) ni en lípidos · EE (0.951 vs 0.472). Tampoco en el RMSE de PLGA · EE (5.7 vs 2.2 %).
   - En lípidos · dH, cuya partición se crea con `set.seed()` justo antes, el r² de **entrenamiento** coincide exactamente con el paper (0.950), pero el de prueba no.
   - Causas posibles, que **no se pueden distinguir** con la información disponible:
     - versiones distintas de los paquetes de R (el autor no las reporta);
     - una versión distinta de la base de datos;
     - una versión del código distinta de la que se compartió.
   - El r² de 0.472 en lípidos · EE del paper es incoherente con su propia validación cruzada (0.929, Tabla S1). El código actual da 0.95.
4. **¿Cuánto de esto es azar? (500 particiones simuladas, Parte C).** El "memorizador", cuyas predicciones se parecen en un 96–99.7 % a las del RF, se evaluó en 500 particiones generadas con el procedimiento del autor:
   - el r² de prueba varía entre **0.14 y 0.49** de ancho (90 % central), según el modelo;
   - **5 de los 6 valores del paper caen dentro de esa variación** (percentiles 22–80). Por ejemplo, el 0.922 de lípidos · dH es simplemente una partición favorable (p76), así que no requiere otra explicación;
   - **el único valor anómalo es lípidos · EE (0.472):** ninguna de las 500 particiones da algo tan bajo, y el 95 % supera 0.75. No es azar;
   - las reproducciones en R y Python caen dentro del rango esperado en los 6 modelos.

   Detalle en `docs/reproduccion_paper.md` §5.3.

### 7.2 Tablas S1 y S2: comparación de algoritmos

- **El código R del autor reproduce bien el SI**, incluidos bagged MARS y bagged MLP. Ejemplos de RMSE de CV en lípidos · dH, R vs paper:
  - random forest: 15.9 vs 16;
  - XGBoost: 16.5 vs 17;
  - bagged MARS: 24.2 vs 24;
  - bagged MLP: 21.2 vs 20.
- **En Python, los modelos lineales coinciden casi exactamente** con el paper y con R (p. ej. lípidos · PdI: 0.083 en los tres). Esto confirma que datos y preprocesamiento son equivalentes.
- **Las diferencias de Python en XGBoost** (lípidos · dH: 31.1 vs 16.5 en R) **y en SVM radial** se deben a la búsqueda aleatoria de solo 10 puntos: R y Python sortean configuraciones distintas. No vienen de los datos ni del algoritmo: con su propio sorteo, R sí reproduce el SI.
- **La conclusión cualitativa se mantiene en las tres fuentes:** ensambles de árboles y kNN son los mejores, y los modelos lineales los peores. Bagged MARS y bagged MLP no son el mejor algoritmo en ningún objetivo, así que excluirlos de Python no cambia las conclusiones.

### 7.3 Conclusión de la reproducción

- **La reimplementación en Python es fiel al código del autor:** con las mismas filas da los mismos resultados (r² ± 0.02), incluso con la rejilla reducida.
- **Las diferencias con la Tabla 2 publicada no vienen de la reimplementación.** En 5 de 6 modelos se explican por **la partición aleatoria**. El valor publicado de lípidos · EE (0.472) es **anómalo**: ninguna partición lo reproduce, ni el propio código del autor.
- **PLGA:** reproducción exitosa en dH y EE (r² ± 0.01 con la partición propia).
- **Lípidos:** parcial, por las razones anteriores.
- **Comparación de algoritmos:** se reproducen las conclusiones. Las diferencias en XGBoost y SVM radial se explican por la búsqueda aleatoria.
- **Repetir la Tabla 2 con el RF real en muchas particiones** se hará dentro de la Parte D, donde es necesario para comparar la partición por medición con la partición por receta (H1).
- Esta reproducción es la **línea base** de la Parte D. La Parte C muestra que, en todas estas particiones, las métricas miden la reproducción de recetas ya vistas.